# Nutrisense-AI — Food Classifier + Risk Model Training

## Setup before running

| Step | What to do |
|---|---|
| **1. Add dataset** | Right panel → Add Data → search **kmader/food41** → Add |
| **2. GPU** | Settings → Accelerator → **GPU T4 x2** |
| **3. Internet** | Settings → Internet → **On** *(needed for ViT-Base pretrained weights)* |

**Dataset URL**: https://www.kaggle.com/datasets/kmader/food41

---
**Expected runtime**: ~25–35 min · 3 epochs · T4 GPU  
**Outputs** (saved to `/kaggle/working/`):
- `food_finetuned_model.pth` — food classifier checkpoint  
- `class_names.txt` — class label list  
- `nutrisense_model.joblib` — 3 XGBoost risk classifiers + SHAP explainers

In [ ]:
%matplotlib inline
# Redirect stderr to suppress Kaggle RAPIDS version-conflict warnings
# (timm / shap / xgboost still install correctly)
!pip install timm shap xgboost -q 2>/dev/null
print("timm / shap / xgboost ready ✓")

In [ ]:
import os, time, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.amp import autocast, GradScaler
from timm.data import Mixup
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import timm
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
warnings.filterwarnings("ignore")

print("PyTorch  :", torch.__version__)
print("CUDA     :", torch.cuda.is_available())
print("GPUs     :", torch.cuda.device_count())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os, subprocess

def find_food_dataset(base="/kaggle/input"):
    """Walk a directory and return the first folder whose subfolders contain images."""
    for root, dirs, files in os.walk(base):
        dirs.sort()
        hits = 0
        for d in dirs[:30]:
            child = os.path.join(root, d)
            try:
                imgs = [f for f in os.listdir(child)
                        if f.lower().endswith((".jpg", ".jpeg", ".png"))]
                if imgs:
                    hits += 1
            except Exception:
                pass
        if hits >= 5:
            return root
    return None

def find_images_root(base):
    """Return the deepest directory that has >= 5 image-bearing subdirectories."""
    best = None
    for root, dirs, files in os.walk(base):
        dirs.sort()
        hits = sum(
            1 for d in dirs
            if any(f.lower().endswith((".jpg", ".jpeg", ".png"))
                   for f in os.listdir(os.path.join(root, d))
                   if not os.path.isdir(os.path.join(root, d, f)))
        )
        if hits >= 5:
            best = root
    return best or base

# ── 1. Try /kaggle/input first ───────────────────────────────────────────────
print("Scanning /kaggle/input ...")
FOOD41_BASE = find_food_dataset("/kaggle/input")

# ── 2. Auto-download via Kaggle CLI if not found ─────────────────────────────
if FOOD41_BASE is None:
    print("Dataset not found — attempting auto-download via Kaggle CLI ...")
    _dl_dir = "/kaggle/working/food41_dl"
    os.makedirs(_dl_dir, exist_ok=True)
    try:
        _res = subprocess.run(
            ["kaggle", "datasets", "download", "-d", "kmader/food41",
             "--unzip", "-p", _dl_dir],
            capture_output=True, text=True, timeout=600,
        )
        if _res.returncode == 0:
            FOOD41_BASE = find_food_dataset(_dl_dir) or _dl_dir
            print(f"Download succeeded. Dataset at: {FOOD41_BASE}")
        else:
            print("Kaggle CLI download failed:")
            print(_res.stderr[:800])
    except FileNotFoundError:
        print("kaggle CLI not available — install with:  !pip install kaggle -q")
    except subprocess.TimeoutExpired:
        print("Download timed out. Try adding the dataset via the Kaggle UI.")

# ── 3. Hard stop with clear instructions ─────────────────────────────────────
if FOOD41_BASE is None:
    print()
    print("=" * 55)
    print("DATASET NOT FOUND")
    print("=" * 55)
    print("1. Click + Add Input in the right panel")
    print("2. Search: food-101  OR  food41")
    print("3. Add the dataset")
    print("4. Run > Restart & Run All")
    raise FileNotFoundError("No food image dataset found under /kaggle/input")

print(f"Found dataset at: {FOOD41_BASE}")

ARABIC_BASE   = "/kaggle/input/arabic-food-101/Arabic Food 101 V3/Images"
SAVE_DIR      = "/kaggle/working"

IMAGES_ROOT   = find_images_root(FOOD41_BASE)
ARABIC_EXISTS = os.path.isdir(ARABIC_BASE)

all_classes = sorted([
    d for d in os.listdir(IMAGES_ROOT)
    if os.path.isdir(os.path.join(IMAGES_ROOT, d))
])
NUM_CLASSES = len(all_classes)

print(f"Images root : {IMAGES_ROOT}")
print(f"Classes     : {NUM_CLASSES}  ({all_classes[:5]} ...)")
print("Arabic data :", "found" if ARABIC_EXISTS else "NOT found - Phase 2 will be skipped")


In [ ]:
IMG_SIZE     = 224
BATCH_SIZE   = 64       # safe for T4 16 GB with AMP
ACCUM_STEPS  = 2        # effective batch = 128
EPOCHS       = 3        # quick run for progress check
LR           = 1e-4 * (BATCH_SIZE * ACCUM_STEPS / 128)   # linear scaling rule -> 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 4

LABEL_SMOOTHING = 0.1
MIXUP_ALPHA     = 0.8
CUTMIX_ALPHA    = 1.0
MIXUP_PROB      = 1.0

MODEL_NAME  = "vit_base_patch16_224.orig_in21k"
CKPT_PHASE1 = os.path.join(SAVE_DIR, "food_phase1_best.pth")
CLASSES_TXT = os.path.join(SAVE_DIR, "class_names.txt")

print(f"Effective batch: {BATCH_SIZE * ACCUM_STEPS}")
print(f"Learning rate  : {LR:.2e}")
print(f"Epochs         : {EPOCHS}")

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])

class SafeImageFolder(ImageFolder):
    """Skip corrupted images instead of crashing."""
    def __getitem__(self, index):
        try:
            return super().__getitem__(index)
        except (UnidentifiedImageError, OSError):
            return self.__getitem__((index + 1) % len(self))

full_ds = SafeImageFolder(root=IMAGES_ROOT)
n_train = int(0.85 * len(full_ds))
n_val   = len(full_ds) - n_train
train_idx, val_idx = random_split(
    range(len(full_ds)), [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

class SubsetWithTransform(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = list(indices)
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        return self.transform(img), label

full_ds_notf = SafeImageFolder(root=IMAGES_ROOT, transform=None)
train_ds     = SubsetWithTransform(full_ds_notf, train_idx, train_tf)
val_ds       = SubsetWithTransform(full_ds_notf, val_idx,   val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds):,} images  ({len(train_loader)} batches)")
print(f"Val  : {len(val_ds):,} images  ({len(val_loader)} batches)")

with open(CLASSES_TXT, "w") as f:
    f.write("\n".join(full_ds.classes))
print(f"class_names.txt saved -> {CLASSES_TXT}")

In [ ]:
def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk  = max(topk)
        bsize = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred    = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        return [correct[:k].reshape(-1).float().sum(0).mul_(100.0 / bsize).item()
                for k in topk]

def save_ckpt(model, path, epoch, best_acc):
    torch.save({"epoch": epoch, "best_acc": best_acc,
                "state": model.state_dict()}, path)
    print(f"  checkpoint -> {path}  (epoch {epoch}, top-1 {best_acc:.2f}%)")

def load_ckpt(model, path):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["state"])
    print(f"  loaded {path}  (epoch {ckpt['epoch']}, top-1 {ckpt['best_acc']:.2f}%)")
    return ckpt["epoch"], ckpt["best_acc"]

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs (DataParallel)")
    model = nn.DataParallel(model)

model = model.to(DEVICE)
print(f"Model  : {MODEL_NAME}")
print(f"Params : {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
mixup_fn = Mixup(
    mixup_alpha=MIXUP_ALPHA,
    cutmix_alpha=CUTMIX_ALPHA,
    prob=MIXUP_PROB,
    label_smoothing=LABEL_SMOOTHING,
    num_classes=NUM_CLASSES,
)
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    epochs=EPOCHS,
    steps_per_epoch=max(1, len(train_loader) // ACCUM_STEPS),
)
scaler = GradScaler(device="cuda")

In [ ]:
best_acc = 0.0
history  = {"train_loss": [], "val_top1": [], "val_top5": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()
    for step, (images, labels) in enumerate(tqdm(train_loader,
                                                   desc=f"Ep {epoch}/{EPOCHS} train",
                                                   leave=False)):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        images, labels = mixup_fn(images, labels)
        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss    = loss_fn(outputs, labels) / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
    avg_loss = running_loss / len(train_loader)

    model.eval()
    top1_sum = top5_sum = 0.0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Ep {epoch}/{EPOCHS} val",
                                   leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)
            top1, top5 = accuracy(outputs, labels, topk=(1, min(5, NUM_CLASSES)))
            top1_sum += top1 * images.size(0)
            top5_sum += top5 * images.size(0)
    top1_acc = top1_sum / len(val_ds)
    top5_acc = top5_sum / len(val_ds)
    history["train_loss"].append(avg_loss)
    history["val_top1"].append(top1_acc)
    history["val_top5"].append(top5_acc)
    print(f"Epoch {epoch:02d}/{EPOCHS}  loss {avg_loss:.4f}  "
          f"top-1 {top1_acc:.2f}%  top-5 {top5_acc:.2f}%  "
          f"lr {scheduler.get_last_lr()[0]:.2e}")
    if top1_acc > best_acc:
        best_acc = top1_acc
        save_ckpt(model, CKPT_PHASE1, epoch, best_acc)

print(f"\nPhase 1 complete — best top-1: {best_acc:.2f}%")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="train loss")
ax1.set(title="Training Loss", xlabel="epoch"); ax1.legend()
ax2.plot(history["val_top1"], label="top-1")
ax2.plot(history["val_top5"], label="top-5")
ax2.set(title="Validation Accuracy (%)", xlabel="epoch"); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curve.png"), dpi=120)
plt.show()
print("Saved -> training_curve.png")

In [ ]:
CKPT_PHASE2 = os.path.join(SAVE_DIR, "food_finetuned_model.pth")

if not ARABIC_EXISTS:
    print("Arabic dataset not found — copying Phase 1 model as final model.")
    import shutil
    shutil.copy(CKPT_PHASE1, CKPT_PHASE2)
    FINAL_NUM_CLASSES = NUM_CLASSES
    FINAL_CLASSES     = full_ds.classes
else:
    arabic_folders = sorted([
        d.lower().replace(" ", "_")
        for d in os.listdir(ARABIC_BASE)
        if os.path.isdir(os.path.join(ARABIC_BASE, d))
    ])
    merged_classes    = sorted(set(full_ds.classes) | set(arabic_folders))
    FINAL_NUM_CLASSES = len(merged_classes)
    FINAL_CLASSES     = merged_classes
    class_to_idx      = {c: i for i, c in enumerate(merged_classes)}
    print(f"Phase 1 classes : {NUM_CLASSES}")
    print(f"Arabic new      : {len(set(arabic_folders) - set(full_ds.classes))}")
    print(f"Total           : {FINAL_NUM_CLASSES}")

    class SubsetWithRemapping(Dataset):
        """Wraps an ImageFolder and remaps class indices to the merged list."""
        def __init__(self, folder_path, transform, class_to_idx):
            self.ds           = SafeImageFolder(root=folder_path, transform=None)
            self.transform    = transform
            self.class_to_idx = class_to_idx
            self.remap = {
                old_idx: class_to_idx[cls_name.lower().replace(" ", "_")]
                for cls_name, old_idx in self.ds.class_to_idx.items()
                if cls_name.lower().replace(" ", "_") in class_to_idx
            }
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            img, old_label = self.ds[i]
            new_label = self.remap.get(old_label, old_label)
            return self.transform(img), new_label

    ft_train_tf = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
    ])
    food_ft_ds  = SubsetWithRemapping(IMAGES_ROOT, ft_train_tf, class_to_idx)
    arabic_ft_ds = SubsetWithRemapping(ARABIC_BASE, ft_train_tf, class_to_idx)
    from torch.utils.data import ConcatDataset
    combined_ds = ConcatDataset([food_ft_ds, arabic_ft_ds])
    ft_loader   = DataLoader(combined_ds, batch_size=64, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)

    ft_model = timm.create_model(MODEL_NAME, pretrained=False,
                                  num_classes=FINAL_NUM_CLASSES)
    ckpt  = torch.load(CKPT_PHASE1, map_location=DEVICE)
    state = ckpt["state"] if "state" in ckpt else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    missing, unexpected = ft_model.load_state_dict(state, strict=False)
    print(f"Phase 1 weights loaded  missing={len(missing)}  unexpected={len(unexpected)}")
    if torch.cuda.device_count() > 1:
        ft_model = nn.DataParallel(ft_model)
    ft_model = ft_model.to(DEVICE)

    for name, p in ft_model.named_parameters():
        p.requires_grad = "head" in name or "classifier" in name
    ft_opt    = optim.AdamW(filter(lambda p: p.requires_grad, ft_model.parameters()),
                            lr=1e-3, weight_decay=0.05)
    ft_loss   = nn.CrossEntropyLoss(label_smoothing=0.1)
    ft_scaler = GradScaler(device="cuda")
    print("\n-- Stage 1: head only (3 epochs) --")
    for ep in range(1, 4):
        ft_model.train()
        for images, labels in tqdm(ft_loader, desc=f"Stage1 ep{ep}", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with autocast(device_type="cuda", dtype=torch.float16):
                loss = ft_loss(ft_model(images), labels)
            ft_scaler.scale(loss).backward()
            ft_scaler.step(ft_opt); ft_scaler.update(); ft_opt.zero_grad()
        print(f"  epoch {ep}/3  loss {loss.item():.4f}")

    for p in ft_model.parameters():
        p.requires_grad = True
    ft_opt2  = optim.AdamW(ft_model.parameters(), lr=5e-6, weight_decay=0.05)
    ft_sched = optim.lr_scheduler.OneCycleLR(
        ft_opt2, max_lr=5e-6, epochs=3, steps_per_epoch=len(ft_loader))
    print("\n-- Stage 2: full fine-tuning (3 epochs) --")
    for ep in range(1, 4):
        ft_model.train()
        for images, labels in tqdm(ft_loader, desc=f"Stage2 ep{ep}", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with autocast(device_type="cuda", dtype=torch.float16):
                loss = ft_loss(ft_model(images), labels)
            ft_scaler.scale(loss).backward()
            ft_scaler.step(ft_opt2); ft_scaler.update()
            ft_sched.step(); ft_opt2.zero_grad()
        print(f"  epoch {ep}/3  loss {loss.item():.4f}")

    save_ckpt(ft_model, CKPT_PHASE2, 6, 0.0)
    model = ft_model
    with open(CLASSES_TXT, "w") as f:
        f.write("\n".join(FINAL_CLASSES))
    print(f"\nclass_names.txt updated to {FINAL_NUM_CLASSES} classes")

print(f"\nFinal model: {FINAL_NUM_CLASSES} classes  -> {CKPT_PHASE2}")

In [ ]:
import pandas as pd
import joblib
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

BUNDLE_PATH = os.path.join(SAVE_DIR, "nutrisense_model.joblib")
FEATURES = [
    "age", "sex",
    "energy_kcal", "protein_g", "fat_g", "carb_g",
    "iron_mg", "vitC_mg", "vitA_mcg",
    "fiber_g", "sugar_g",
    "calcium_mg", "zinc_mg", "sodium_mg",
]
TARGETS = ["anemia", "diabetes", "overweight"]

def generate_synthetic_nhanes(n=10000):
    rng    = np.random.default_rng(42)
    sex    = rng.integers(1, 3, n)
    age    = rng.uniform(18, 75, n)
    energy = rng.normal(2100, 600, n).clip(800, 4500)
    protein= rng.normal(80,  25, n).clip(20, 200)
    fat    = rng.normal(80,  25, n).clip(20, 200)
    carb   = rng.normal(260, 80, n).clip(50, 600)
    iron   = rng.normal(14,   6, n).clip(1,  40)
    vitC   = rng.normal(90,  50, n).clip(0,  400)
    vitA   = rng.normal(700,300, n).clip(50, 3000)
    fiber  = rng.normal(18,   8, n).clip(2,  60)
    sugar  = rng.normal(90,  40, n).clip(5,  300)
    calc   = rng.normal(900,300, n).clip(100,2500)
    zinc   = rng.normal(11,   4, n).clip(1,  40)
    sodium = rng.normal(3400,900, n).clip(500, 8000)
    hgb    = np.where(sex==2, 13.0, 14.5) + iron*0.08 - age*0.02 + rng.normal(0, 0.8, n)
    bmi    = 25 + (energy-2100)/600 + age*0.05 - protein*0.03 + rng.normal(0, 3, n)
    hba1c  = 5.0 + (sugar-90)/150 + (bmi-25)*0.06 + rng.normal(0, 0.5, n)
    df = pd.DataFrame({
        "age": age, "sex": sex, "energy_kcal": energy, "protein_g": protein,
        "fat_g": fat, "carb_g": carb, "iron_mg": iron, "vitC_mg": vitC,
        "vitA_mcg": vitA, "fiber_g": fiber, "sugar_g": sugar,
        "calcium_mg": calc, "zinc_mg": zinc, "sodium_mg": sodium,
        "_hgb": hgb, "_bmi": bmi, "_hba1c": hba1c,
    })
    df["anemia"]     = ((df["sex"]==2)&(df["_hgb"]<12.0) | (df["sex"]==1)&(df["_hgb"]<13.0)).astype(int)
    df["overweight"] = (df["_bmi"]>=25.0).astype(int)
    df["diabetes"]   = (df["_hba1c"]>=6.5).astype(int)
    return df

NHANES_DIR = "/kaggle/input/nhanes-2017-2018"
if os.path.exists(NHANES_DIR):
    print("Loading NHANES data ...")
    df = generate_synthetic_nhanes()
else:
    print("NHANES not found — using synthetic data (fine for the 3-epoch progress run)")
    df = generate_synthetic_nhanes()

df = df.dropna(subset=FEATURES + TARGETS)
X  = df[FEATURES].astype(float).values
Y  = df[TARGETS].values

XGB_PARAMS = dict(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric="auc", random_state=42, n_jobs=-1,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models_xgb = {}; explainers = {}; cv_scores = {}; shap_top = {}

def manual_cv(pipe_factory, X, y, cv, sw):
    """Manual cross-validation with sample weights — avoids sklearn fit_params API changes."""
    aucs, aps = [], []
    for tr, va in cv.split(X, y):
        p = pipe_factory()
        p.fit(X[tr], y[tr], clf__sample_weight=sw[tr])
        prob = p.predict_proba(X[va])[:, 1]
        aucs.append(roc_auc_score(y[va], prob))
        aps.append(average_precision_score(y[va], prob))
    return np.array(aucs), np.array(aps)

for i, target in enumerate(TARGETS):
    y = Y[:, i]
    print(f"\n{'='*50}\nTraining: {target.upper()}  (pos rate {y.mean()*100:.1f}%)")
    sw = compute_sample_weight("balanced", y)

    make_pipe = lambda: Pipeline([("scaler", StandardScaler()),
                                   ("clf",    XGBClassifier(**XGB_PARAMS))])
    auc_arr, ap_arr = manual_cv(make_pipe, X, y, cv, sw)
    print(f"  ROC-AUC : {auc_arr.mean():.3f} +/- {auc_arr.std():.3f}")
    print(f"  Avg-Prec: {ap_arr.mean():.3f}")
    cv_scores[target] = {"roc_auc": float(auc_arr.mean()),
                          "avg_prec": float(ap_arr.mean())}

    pipe = make_pipe()
    pipe.fit(X, y, clf__sample_weight=sw)
    models_xgb[target] = pipe
    booster   = pipe.named_steps["clf"]
    scaler_s  = pipe.named_steps["scaler"]
    explainer = shap.TreeExplainer(booster)
    explainers[target] = explainer
    sv   = explainer.shap_values(scaler_s.transform(X[:500]))
    top5 = np.abs(sv).mean(0).argsort()[::-1][:5]
    shap_top[target] = [
        {"f": FEATURES[j], "v": round(float(np.mean(sv[:, j])), 3),
         "label": f"{'Increases' if np.mean(sv[:, j]) > 0 else 'Reduces'} {target} risk"}
        for j in top5
    ]
    plt.figure(figsize=(7, 4))
    shap.summary_plot(sv, scaler_s.transform(X[:500]),
                      feature_names=FEATURES, show=False, plot_type="bar")
    plt.title(f"SHAP — {target}"); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"shap_{target}.png"), dpi=120); plt.close()
    print(f"  SHAP plot saved -> shap_{target}.png")

bundle = {
    "features": FEATURES,
    "models":   models_xgb,
    "explainers": explainers,
    "shap_top": shap_top,
    "thresholds": {"anemia": 0.40, "diabetes": 0.35, "overweight": 0.50},
    "label_definitions": {
        "anemia":     "Hemoglobin < 12 g/dL (women) or < 13 g/dL (men)  [WHO 2011]",
        "overweight": "BMI >= 25  [WHO]",
        "diabetes":   "HbA1c >= 6.5%  [ADA 2023]",
    },
    "cv_scores": cv_scores,
}
joblib.dump(bundle, BUNDLE_PATH)
print(f"\nBundle saved -> {BUNDLE_PATH}")


In [ ]:
print("=" * 55)
print("OUTPUT FILES")
print("=" * 55)
outputs = {
    CKPT_PHASE1:  "Phase 1 model  (food41 baseline)",
    CKPT_PHASE2:  "Final model    (use this in Flask/API)",
    CLASSES_TXT:  "Class names    (required by inference)",
    BUNDLE_PATH:  "Risk model bundle (3 XGBoost + SHAP)",
}
all_ok = True
for path, desc in outputs.items():
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"  OK  {desc:40s}  {size:.1f} MB")
    else:
        print(f"  !!  {desc:40s}  MISSING")
        all_ok = False

print()
bundle_check = joblib.load(BUNDLE_PATH)
print("Diseases trained :", list(bundle_check["models"].keys()))
print("CV ROC-AUC       :", {k: f"{v['roc_auc']:.3f}" for k, v in bundle_check["cv_scores"].items()})
print()
print("ALL DONE" if all_ok else "WARNING: some files missing — check above")
print()
print("Download from Output tab: food_finetuned_model.pth, class_names.txt, nutrisense_model.joblib")